# 1 — Reformat GWHD 2021 → crop-counter CVAT 1.1

Convert the **Global Wheat Head Detection 2021** dataset (`data/gwhd_2021`) into the
point-annotation layout that `crop-counter` expects (see `crop-counter/examples/data`).

**Source format** — three CSVs (`competition_train.csv`, `competition_val.csv`,
`competition_test.csv`) with columns `image_name, BoxesString, domain`.
`BoxesString` is a `;`-separated list of boxes, each `x1 y1 x2 y2` (top-left / bottom-right,
space-separated), or the literal `no_box` for an empty image. Images are 1024×1024 PNGs
under `images/`.

**Target format** — one folder per split, each holding `images/` and a
`annotations.xml` ("CVAT for images 1.1"). Every image is an `<image>` element carrying
one `<points label=... points="x,y" />` per annotation.

**Conversion rule** — each bounding box becomes its **center point**
`((x1+x2)/2, (y1+y2)/2)`, labelled `Wheat` (the label `crop-counter` counts by default).
`no_box` images are written as `<image>` elements with zero points, so image counts are
preserved. The `test` split is carried through in the same format alongside `train`/`val`.

In [ ]:
from pathlib import Path
import shutil
import xml.etree.ElementTree as ET

import pandas as pd
from PIL import Image

# --- paths ---------------------------------------------------------------
SRC = Path(r"path\to\WheatHead\data\gwhd_2021")
DST = Path(r"path\to\WheatHead\data\gwhd_2021_reformat")
SRC_IMAGES = SRC / "images"

# GWHD CSV  ->  crop-counter split folder
SPLITS = {
    "competition_train.csv": "train",
    "competition_val.csv": "val",
    "competition_test.csv": "test",
}

LABEL = "Wheat"       # every GWHD head is a wheat head
COPY_IMAGES = True     # set False to (re)write only the XML and skip copying

assert SRC_IMAGES.is_dir(), f"missing image folder: {SRC_IMAGES}"
DST.mkdir(parents=True, exist_ok=True)
print("source:", SRC)
print("target:", DST)

In [ ]:
def boxes_to_points(boxes_string: str):
    """Parse a GWHD `BoxesString` into a list of (cx, cy) box-center points.

    `boxes_string` is `x1 y1 x2 y2;x1 y1 x2 y2;...`, or the literal `no_box`
    (also handles empty / NaN cells) which yields an empty list.
    """
    s = "" if boxes_string is None else str(boxes_string).strip()
    if s == "" or s.lower() == "no_box":
        return []
    points = []
    for box in s.split(";"):
        box = box.strip()
        if not box:
            continue
        x1, y1, x2, y2 = (float(v) for v in box.split())
        points.append(((x1 + x2) / 2.0, (y1 + y2) / 2.0))
    return points


def build_cvat_xml(records):
    """Build a CVAT-1.1 <annotations> tree from (name, width, height, points)."""
    root = ET.Element("annotations")
    ET.SubElement(root, "version").text = "1.1"
    for idx, (name, width, height, points) in enumerate(records):
        image_el = ET.SubElement(
            root, "image",
            id=str(idx), name=name, width=str(width), height=str(height),
        )
        for (x, y) in points:
            ET.SubElement(
                image_el, "points",
                label=LABEL, occluded="0",
                points=f"{x:.2f},{y:.2f}", z_order="0",
            )
    ET.indent(root, space="  ")  # pretty-print (Python 3.9+)
    return ET.ElementTree(root)

In [ ]:
def reformat_split(csv_name: str, split: str):
    """Convert one GWHD CSV into a crop-counter split folder."""
    df = pd.read_csv(SRC / csv_name)
    out_dir = DST / split
    out_images = out_dir / "images"
    out_images.mkdir(parents=True, exist_ok=True)

    records = []
    n_points = 0
    n_empty = 0
    for i, row in enumerate(df.itertuples(index=False), start=1):
        name = row.image_name
        src_img = SRC_IMAGES / name
        if not src_img.is_file():
            raise FileNotFoundError(f"{csv_name}: image not found -> {src_img}")

        with Image.open(src_img) as im:
            width, height = im.size

        points = boxes_to_points(row.BoxesString)
        n_points += len(points)
        n_empty += (len(points) == 0)
        records.append((name, width, height, points))

        if COPY_IMAGES:
            shutil.copy2(src_img, out_images / name)

        if i % 500 == 0:
            print(f"  {split}: {i}/{len(df)} images processed")

    xml_path = out_dir / "annotations.xml"
    build_cvat_xml(records).write(xml_path, encoding="utf-8", xml_declaration=True)

    print(
        f"[{split}] {len(records)} images, {n_points} points, "
        f"{n_empty} empty  ->  {xml_path}"
    )
    return {"split": split, "images": len(records), "points": n_points, "empty": n_empty}


summary = [reformat_split(csv, split) for csv, split in SPLITS.items()]
pd.DataFrame(summary)

## Verify

Round-trip the output through `crop-counter`'s own CVAT loader and check that the
reloaded point counts match what we wrote.

In [ ]:
import sys

sys.path.insert(0, str(Path(r"D:\Projects\WheatHead\crop-counter\src")))
from cropcounter.crop_dataset import parse_cvat_1_1  # noqa: E402

for row in summary:
    split = row["split"]
    recs = parse_cvat_1_1(DST / split / "annotations.xml")
    total = sum(len(r.points) for r in recs)
    ok = (len(recs) == row["images"]) and (total == row["points"])
    print(
        f"[{split}] reloaded {len(recs)} images / {total} points  "
        f"{'OK' if ok else 'MISMATCH'}"
    )
    assert ok, f"round-trip mismatch for split={split}"

In [ ]:
# Peek at the first image element of the train split
print((DST / "train" / "annotations.xml").read_text(encoding="utf-8")[:900])